# BBQ data exploration

This notebook loads the BBQ data for our 4 categories (Gender_identity, Race_ethnicity, Age, SES),
adds the stereotype target from BBQ's metadata file, and turns every question into 3 rows
(one per answer option) with our labels. The column definitions are in `docs/schema.md`.

## 1. Setup

Imports, the data URLs, and the list of categories we use. Everything is read straight from the
BBQ GitHub repo, so there's nothing to download first.

In [ ]:
import pandas as pd

base = "https://raw.githubusercontent.com/nyu-mll/BBQ/main/data/"
meta_url = "https://raw.githubusercontent.com/nyu-mll/BBQ/main/analysis_scripts/additional_metadata.csv"
categories = ["Gender_identity", "Race_ethnicity", "Age", "SES"]

# columns we use to match BBQ rows to rows in the metadata file
keys = ["category", "question_index", "example_id"]

## 2. Look at one category: Gender_identity

Before running all 4 categories, we look at one closely. Each BBQ template (`question_index`)
should come in 4 versions: an ambiguous or disambiguated context, combined with a negative or
non-negative question. We check that every template has all 4, then print one example row to see
what the fields look like.

In [ ]:
df = pd.read_json(base + "Gender_identity.jsonl", lines=True)
print(df.shape)

In [ ]:
# each template should have 2 context conditions x 2 question polarities
versions = df.groupby("question_index")[["context_condition", "question_polarity"]].nunique()
print("templates:", len(versions))
print("all templates have 4 versions:", (versions == 2).all().all())

In [ ]:
row = df.iloc[0]
print(row["context"])
print(row["question"])
print(row["ans0"], "|", row["ans1"], "|", row["ans2"])
print("correct:", row["label"])
print("answer_info:", row["answer_info"])
print("stereotyped_groups:", row["additional_metadata"]["stereotyped_groups"])

## 3. Load the metadata file and join it to the data

To label bias we need to know which answer option is the stereotype target. The data file has
`stereotyped_groups`, but it uses short labels like "F", while `answer_info` uses words like
"woman", so matching them directly is messy. BBQ's `additional_metadata.csv` has a `target_loc`
column that gives the target's answer index (0, 1 or 2) directly, so we use that.

The metadata file has some duplicate rows. If we join without removing them, those questions get
matched twice and we end up with extra rows. So we check the duplicates (they all have the same
`target_loc`, so it's safe to keep one), remove them, and then join.

In [ ]:
meta = pd.read_csv(meta_url)
print(meta.columns)

# keep only the columns we need from the metadata file
meta_small = meta[keys + ["target_loc"]]

In [ ]:
dupes = meta_small[meta_small.duplicated(subset=keys, keep=False)]
print(len(dupes), "rows involved in duplicates,", meta_small.duplicated(subset=keys).sum(), "extra copies")
print(dupes["category"].value_counts())

conflicts = dupes.groupby(keys)["target_loc"].nunique()
print((conflicts > 1).sum(), "pairs with different target_loc")

In [ ]:
# what happens if we join without removing the duplicates
print("join with duplicates:", len(df.merge(meta_small, on=keys, how="left")), "rows (data has", len(df), ")")

meta_small = meta_small.drop_duplicates(subset=keys)

# add target_loc to each BBQ row by matching on the key columns
merged = df.merge(meta_small, on=keys, how="left")
print("join after removing duplicates:", len(merged), "rows")

# check our example row
print(merged.iloc[0][["question", "label", "target_loc"]])

## 4. Rows with missing target_loc

Some questions don't get a `target_loc` from the join. Here we look at which ones they are. Without
a target we can't tell which answer follows the stereotype, so we drop these rows.

In [ ]:
missing = merged[merged["target_loc"].isna()]
print(len(missing), "rows missing target_loc out of", len(merged))
missing[["example_id", "question_index", "context_condition", "question_polarity", "question"]]

In [ ]:
merged = merged.dropna(subset=["target_loc"])
merged["target_loc"] = merged["target_loc"].astype(int)
print(len(merged), "rows left")

## 5. Build the 3-rows-per-question table

Each question becomes 3 rows, one per answer option. For each option we mark:

- `is_correct`: it's the right answer
- `is_biased`: it's wrong and follows the stereotype
- `anti_stereotype`: it's wrong and goes against the stereotype

The full rule is in `docs/schema.md`. The code is wrapped in a function, `build_table`, so the
exact same code runs for all 4 categories in the next section.

Then we check it on the secretary example from section 2. In an ambiguous context the correct
answer is "Not enough information", "The woman" should be biased, and "The man" should be
anti-stereotype.

In [ ]:
def find_unknown(info):
    # returns the index (0, 1, or 2) of the "unknown" answer option
    for i in range(3):
        if info[f"ans{i}"][1] == "unknown":
            return i
    return None


def build_table(m):
    # turns each question in m into 3 rows, one per answer option
    rows = []
    for _, q in m.iterrows():
        unk = find_unknown(q["answer_info"])
        target = q["target_loc"]
        non_target = [i for i in range(3) if i not in (unk, target)][0]

        # which option follows the stereotype for this question
        stereo = target if q["question_polarity"] == "neg" else non_target

        for i in range(3):
            is_correct = int(i == q["label"])
            rows.append({
                "row_id": f"{q['category']}_{q['example_id']}_ans{i}",
                "example_id": q["example_id"],
                "template_id": f"{q['category']}_{q['question_index']}",
                "category": q["category"],
                "context_condition": q["context_condition"],
                "question_polarity": q["question_polarity"],
                "context": q["context"],
                "question": q["question"],
                "answer_text": q[f"ans{i}"],
                "answer_idx": i,
                "is_correct": is_correct,
                "is_biased": int(not is_correct and i == stereo),
                "anti_stereotype": int(not is_correct and i != stereo and i != unk),
            })
    return pd.DataFrame(rows)

In [ ]:
table = build_table(merged)

print(len(table), "rows (should be", 3 * len(merged), ")")
print("share biased:", table["is_biased"].mean())
table[table["example_id"] == merged.iloc[0]["example_id"]][["answer_text", "is_correct", "is_biased", "anti_stereotype"]]

## 6. Run everything for all 4 categories

Same steps as above, for Gender_identity, Race_ethnicity, Age and SES. For each one we print the
checks: number of templates, whether every template has all 4 versions, how many rows were missing
`target_loc`, whether an "unknown" option was found in every question, the final row count, and the
share of biased rows. At the end all 4 tables are stacked into one `table`.

In [ ]:
all_tables = []

for cat in categories:
    d = pd.read_json(base + cat + ".jsonl", lines=True)
    versions = d.groupby("question_index")[["context_condition", "question_polarity"]].nunique()

    m = d.merge(meta_small, on=keys, how="left")
    n_missing = m["target_loc"].isna().sum()
    m = m.dropna(subset=["target_loc"])
    m["target_loc"] = m["target_loc"].astype(int)

    no_unknown = m["answer_info"].apply(find_unknown).isna().sum()

    t = build_table(m)
    all_tables.append(t)

    print(cat)
    print("  templates:", d["question_index"].nunique())
    print("  all templates have 4 versions:", (versions == 2).all().all())
    print("  missing target_loc (dropped):", n_missing)
    print("  no unknown found:", no_unknown)
    print("  rows:", len(t), "| share biased:", round(t["is_biased"].mean(), 3))

table = pd.concat(all_tables, ignore_index=True)
print("total rows:", len(table))

## 7. Summary of findings

- Every template in all 4 categories has all 4 versions (ambig/disambig × neg/nonneg).
- Templates: Gender_identity 50, Race_ethnicity 50, Age 25, SES 25.
- The metadata file has 64 duplicate Gender_identity rows, all with the same `target_loc`. We
  remove them before the join; without that, the join creates extra rows.
- 16 Gender_identity questions (template 13, "Who is really emotional?" / "Who is really logical?",
  4 full sets) have no `target_loc`, so they were dropped.
- An "unknown" answer option was found in every question.
- Final rows: Gender_identity 16968, Race_ethnicity 20640, Age 11040, SES 20592, total 69240.
- The share of biased rows is exactly 0.25 in every category. That's what we expect: 1/3 of the
  options in ambiguous contexts, and 1/6 on average in disambiguated ones.
- Age and SES only have 25 templates each, so their test sets will be small (about 4 templates).

More details are in `docs/data_exploration.md`.